# Lab 8: English → Indic NMT with OpenNMT-py (ILCI subset of BPCC)
Length-binned train/test splits, Transformer NMT, bin-wise BLEU (sacreBLEU).

**Local run on Mac (Apple M4)** - OpenNMT-py has no MPS support, so training runs on the **CPU** (the M4 handles a small Transformer fine; expect tens of minutes, not seconds).

Setup (once, in a terminal), using Python 3.10-3.12 (OpenNMT-py needs torch < 2.3, which has no Python 3.13 build):
```
python3.11 -m venv .venv && source .venv/bin/activate
pip install jupyter ipykernel
python -m ipykernel install --user --name lab8
```
Then open this notebook with the `lab8` kernel. Also:
1. BPCC is *gated*: accept the terms at https://huggingface.co/datasets/ai4bharat/BPCC
2. Make a HF access token: `export HF_TOKEN=hf_xxx` before launching Jupyter, or paste it when prompted.
3. Set `TGT_LANG` below to **your native language** (default Hindi).

In [ ]:
# ---- CONFIG ----
TGT_LANG   = "hin_Deva"     # your native language (FLORES code). e.g. ben_Beng, tam_Taml, tel_Telu, mar_Deva, guj_Gujr, kan_Knda, mal_Mlym, pan_Guru, ory_Orya, urd_Arab
SRC_LANG   = "eng_Latn"
TEST_FRAC  = 0.10           # per-bin test fraction
DEV_SIZE   = 1000           # validation sentences taken (stratified) from train
VOCAB_SIZE = 8000           # joint SentencePiece BPE vocab
TRAIN_STEPS = 5000          # CPU run; raise (e.g. 10000-20000) if you have time
MAX_WORDS  = 100            # drop pairs longer than this (source words)
SEED = 42
import os
WORK = os.path.abspath("./lab8_work")   # all outputs go here
os.makedirs(WORK, exist_ok=True)

: 

In [ ]:
%pip install -q OpenNMT-py sentencepiece sacrebleu huggingface_hub pyyaml pandas pyarrow matplotlib
import sys, torch, onmt
print(sys.version.split()[0], '| torch', torch.__version__, '| onmt', onmt.__version__)
print('MPS available:', torch.backends.mps.is_available(), '(OpenNMT-py trains on CPU regardless)')
print('CPU threads:', torch.get_num_threads())

## 1. Load the ILCI subset

In [ ]:
import os, glob, json, random, re
import numpy as np, pandas as pd
from huggingface_hub import snapshot_download, login
random.seed(SEED); np.random.seed(SEED)

import getpass
HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass("Hugging Face token (must have accepted BPCC terms): ")
assert HF_TOKEN
login(HF_TOKEN)

local = snapshot_download("ai4bharat/BPCC", repo_type="dataset",
                          allow_patterns=["*ilci*", "*ILCI*"], token=HF_TOKEN)
files = [f for f in glob.glob(local + "/**/*", recursive=True)
         if os.path.isfile(f) and f.lower().endswith((".json", ".jsonl", ".csv", ".tsv", ".parquet"))]
print(len(files), "ILCI files found")
for f in files[:30]: print(" ", f.replace(local, ""))

In [ ]:
def read_any(f):
    fl = f.lower()
    if fl.endswith(".parquet"): return pd.read_parquet(f)
    if fl.endswith(".csv"):     return pd.read_csv(f)
    if fl.endswith(".tsv"):     return pd.read_csv(f, sep="\t")
    if fl.endswith(".jsonl"):   return pd.read_json(f, lines=True)
    try:    return pd.read_json(f)
    except ValueError: return pd.read_json(f, lines=True)

def find_col(df, key):
    for c in df.columns:
        if c.lower() == key: return c
    for c in df.columns:
        if key in c.lower(): return c
    return None

pairs = []
for f in files:
    try: df = read_any(f)
    except Exception as e: print("skip", f, e); continue
    sl, tl = find_col(df, "src_lang"), find_col(df, "tgt_lang")
    sc, tc = find_col(df, "src"), find_col(df, "tgt")
    if sc is None or tc is None: print("no src/tgt cols in", f, list(df.columns)); continue
    if sl and tl:   # language columns present -> filter, handle both orientations
        a = df[(df[sl] == SRC_LANG) & (df[tl] == TGT_LANG)][[sc, tc]].values.tolist()
        b = df[(df[sl] == TGT_LANG) & (df[tl] == SRC_LANG)][[tc, sc]].values.tolist()
        pairs += a + b
    elif TGT_LANG in f:  # language encoded in the path only
        pairs += df[[sc, tc]].values.tolist()
print("raw pairs:", len(pairs))
assert pairs, "No pairs found - check TGT_LANG and the file listing above."

In [ ]:
# clean: strip, drop empty / duplicate source / absurd length ratios / very long
seen, data = set(), []
for s, t in pairs:
    if not isinstance(s, str) or not isinstance(t, str): continue
    s, t = " ".join(s.split()), " ".join(t.split())
    if not s or not t or s in seen: continue
    ls, lt = len(s.split()), len(t.split())
    if ls > MAX_WORDS or lt > MAX_WORDS * 2: continue
    if not (1/3 <= (lt / ls) <= 3.5): continue
    seen.add(s); data.append((s, t))
df = pd.DataFrame(data, columns=["src", "tgt"])
df["len"] = df.src.str.split().str.len()
print(len(df), "clean pairs"); df.head()

## 2. Length-binned splits (4 bins on English source word count)
| bin | rule |
|---|---|
| bin1 | len ≤ 10 |
| bin2 | 10 < len ≤ 20 |
| bin3 | 20 < len < 30 |
| bin4 | len ≥ 30 |

(The assignment's open intervals leave len = 20 undefined; it is placed in bin2.)  
Test and train are both built **stratified by bin**, so every split contains all four length ranges.

In [ ]:
def to_bin(n):
    if n <= 10: return "bin1"
    if n <= 20: return "bin2"
    if n < 30:  return "bin3"
    return "bin4"
df["bin"] = df.len.map(to_bin)
print(df.bin.value_counts().sort_index())

rng = np.random.RandomState(SEED)
test_idx = []
for b, g in df.groupby("bin"):
    n = max(1, int(round(len(g) * TEST_FRAC)))
    test_idx += list(rng.choice(g.index, size=n, replace=False))
test = df.loc[test_idx].reset_index(drop=True)
rest = df.drop(index=test_idx)

dev_idx = []
for b, g in rest.groupby("bin"):
    n = max(1, int(round(DEV_SIZE * len(g) / len(rest))))
    dev_idx += list(rng.choice(g.index, size=min(n, len(g) - 1), replace=False))
dev   = rest.loc[dev_idx].reset_index(drop=True)
train = rest.drop(index=dev_idx).sample(frac=1, random_state=SEED).reset_index(drop=True)

print(pd.DataFrame({"train": train.bin.value_counts(), "dev": dev.bin.value_counts(), "test": test.bin.value_counts()}).sort_index())

## 3. SentencePiece (joint BPE) and OpenNMT data files

In [ ]:
import sentencepiece as spm
D = f"{WORK}/data"; os.makedirs(D, exist_ok=True)

with open(f"{D}/spm_train.txt", "w") as f:
    for s, t in zip(train.src, train.tgt): f.write(s + "\n" + t + "\n")
spm.SentencePieceTrainer.train(input=f"{D}/spm_train.txt", model_prefix=f"{D}/spm", vocab_size=VOCAB_SIZE,
                               model_type="bpe", character_coverage=1.0, input_sentence_size=2000000,
                               shuffle_input_sentence=True)
sp = spm.SentencePieceProcessor(model_file=f"{D}/spm.model")

def write(name, series):
    with open(f"{D}/{name}", "w") as f:
        for x in series: f.write(" ".join(sp.encode(x, out_type=str)) + "\n")

for nm, d in [("train", train), ("dev", dev), ("test", test)]:
    write(f"{nm}.src.sp", d.src); write(f"{nm}.tgt.sp", d.tgt)
    d.to_csv(f"{D}/{nm}.csv", index=False)       # raw text + bin kept for scoring
print(open(f"{D}/train.src.sp").readline())

## 4. OpenNMT-py: build vocab + train a small Transformer

In [ ]:
import yaml
cfg = {
 "save_data": f"{WORK}/run/onmt",
 "src_vocab": f"{WORK}/run/vocab.shared", "tgt_vocab": f"{WORK}/run/vocab.shared",
 "share_vocab": True, "src_vocab_size": VOCAB_SIZE + 10, "tgt_vocab_size": VOCAB_SIZE + 10,
 "overwrite": True, "save_model": f"{WORK}/run/model",
 "data": {"corpus_1": {"path_src": f"{D}/train.src.sp", "path_tgt": f"{D}/train.tgt.sp"},
          "valid":    {"path_src": f"{D}/dev.src.sp",   "path_tgt": f"{D}/dev.tgt.sp"}},
 "seed": SEED,   # no gpu_ranks -> CPU training
 "optim": "adam", "learning_rate": 2, "warmup_steps": 1500,
 "decay_method": "noam", "adam_beta2": 0.998, "max_grad_norm": 0, "label_smoothing": 0.1,
 "param_init": 0, "param_init_glorot": True, "normalization": "tokens",
 "encoder_type": "transformer", "decoder_type": "transformer", "position_encoding": True,
 "enc_layers": 3, "dec_layers": 3, "heads": 4, "hidden_size": 256, "word_vec_size": 256,
 "transformer_ff": 1024, "dropout_steps": [0], "dropout": [0.3], "attention_dropout": [0.1],
 "batch_type": "tokens", "batch_size": 3072, "valid_batch_size": 2048, "accum_count": [2], "accum_steps": [0],
 "train_steps": TRAIN_STEPS, "valid_steps": 500, "save_checkpoint_steps": 500, "keep_checkpoint": 3,
 "early_stopping": 6, "early_stopping_criteria": "ppl", "report_every": 100,
}
os.makedirs(f"{WORK}/run", exist_ok=True)
yaml.safe_dump(cfg, open(f"{WORK}/run/config.yaml", "w"), sort_keys=False)

In [ ]:
import subprocess, shutil
BIN = os.path.dirname(sys.executable)
def onmt(cmd, *args, show=None):
    exe = shutil.which(cmd, path=BIN + os.pathsep + os.environ.get("PATH", "")) or cmd
    p = subprocess.Popen([exe, *args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if show is None or re.search(show, line): print(line.rstrip())
    assert p.wait() == 0, f"{cmd} failed"

onmt("onmt_build_vocab", "-config", f"{WORK}/run/config.yaml", "-n_sample", "-1", show=r"Counter|vocab|Error")

In [ ]:
onmt("onmt_train", "-config", f"{WORK}/run/config.yaml", show=r"Step \d+/.*(100|00)/|Validation|Stalled|Early|Error|error")

## 5. Translate the test set

In [ ]:
ckpts = sorted(glob.glob(f"{WORK}/run/model_step_*.pt"), key=lambda p: int(re.findall(r"step_(\d+)", p)[0]))
CKPT = ckpts[-1]; print("using", CKPT)
onmt("onmt_translate", "-model", CKPT, "-src", f"{D}/test.src.sp", "-output", f"{D}/test.pred.sp",
     "-beam_size", "5", "-batch_size", "32", "-batch_type", "sents", "-max_length", "256", show=r"PRED AVG|Error")

## 6. Bin-wise BLEU (sacreBLEU)

In [ ]:
import sacrebleu
hyps = [sp.decode(l.split()) for l in open(f"{D}/test.pred.sp").read().split("\n")[:len(test)]]
test["hyp"] = hyps
refs = test.tgt.tolist()

rows = []
for b in ["bin1", "bin2", "bin3", "bin4", "ALL"]:
    g = test if b == "ALL" else test[test.bin == b]
    bleu = sacrebleu.corpus_bleu(g.hyp.tolist(), [g.tgt.tolist()])
    chrf = sacrebleu.corpus_chrf(g.hyp.tolist(), [g.tgt.tolist()])
    rows.append(dict(bin=b, n_sents=len(g), avg_src_len=round(g.len.mean(), 1),
                     BLEU=round(bleu.score, 2), chrF=round(chrf.score, 2),
                     len_ratio_hyp_ref=round(sum(len(h.split()) for h in g.hyp) / sum(len(r.split()) for r in g.tgt), 3),
                     BP=round(bleu.bp, 3)))
res = pd.DataFrame(rows); print(res.to_string(index=False))
print("\nSignature:", sacrebleu.BLEU().get_signature())
res.to_csv(f"{WORK}/binwise_bleu.csv", index=False)
test.to_csv(f"{WORK}/test_predictions.csv", index=False)

In [ ]:
import matplotlib.pyplot as plt
r = res[res.bin != "ALL"]
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].bar(r.bin, r.BLEU); ax[0].set_title("BLEU by length bin"); ax[0].set_ylabel("BLEU")
ax[1].bar(r.bin, r.len_ratio_hyp_ref); ax[1].axhline(1, color="k", ls="--"); ax[1].set_title("hyp/ref length ratio")
plt.tight_layout(); plt.savefig(f"{WORK}/binwise_bleu.png", dpi=150); plt.show()

hi, lo = r.loc[r.BLEU.idxmax()], r.loc[r.BLEU.idxmin()]
print(f"Overall BLEU {res[res.bin=='ALL'].BLEU.item()}. Best bin: {hi.bin} ({hi.BLEU}); worst bin: {lo.bin} ({lo.BLEU}).")
print("BLEU trend with length:", " -> ".join(f"{b}:{s}" for b, s in zip(r.bin, r.BLEU)))
print("Brevity (hyp/ref):", " ".join(f"{b}:{s}" for b, s in zip(r.bin, r.len_ratio_hyp_ref)))
# a few examples per bin
for b in ["bin1", "bin2", "bin3", "bin4"]:
    x = test[test.bin == b].sample(1, random_state=1).iloc[0]
    print(f"\n[{b}] SRC: {x.src}\n REF: {x.tgt}\n HYP: {x.hyp}")

## 7. Analysis (fill in after seeing your numbers)
Points to discuss, using the table/plot above:
- **Trend across bins:** which bin has the highest/lowest BLEU, and is the trend monotonic in length?
- **Short sentences (bin1):** few n-grams, so BLEU is noisy and sensitive to a single word/lexical choice; often higher or lower depending on domain (ILCI short sentences are often headings/phrases).
- **Long sentences (bin4):** typical NMT degradation from long-range dependencies, attention dilution and under-translation. Check the `len_ratio_hyp_ref` / brevity penalty: a ratio < 1 means truncated or dropped content.
- **Data effect:** bins have different training-set sizes (see split table); the sparse bin usually performs worse. Small corpora and a small Transformer also limit absolute BLEU.
- **Metric caveat:** BLEU is computed with default 13a tokenization on Indic text; chrF is included as a more robust complement.
- **Possible improvements:** more training steps/larger model, back-translation, fine-tuning a pretrained model (e.g. IndicTrans2), length-balanced training, beam/length penalty tuning.